### Converting multi-span data to BIO format - TherapistQA dataset

TherapistQA key findings:

- 932 non-distorted records (no spans, all-O tokens)
- 1,391 single-span records (span_match_type == 'exact')
- 203 multi-span records (sorted, no overlaps — clean to handle)
- Multi-span records: 177 have 2 spans, 24 have 3, 1 has 4, 1 has 5
- 10 distortion categories + No Distortion
- All multi-spans share the same dominant_distortion label (one row = one record), so when applying BIO category tags, all spans in a record get the same category label

Token-level rule: a token gets B- if it's the first token whose offsets fall inside any span and the previous token is outside; I- if it's a continuation (inside); O otherwise (outside). Each span in all_spans independently produces a B at its start token, and I's after — handled by iterating spans in sorted order.
Output format: parquet (preserves list-of-lists token/tag structure cleanly) plus a JSONL fallback for human inspection.

In [1]:
import json
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

PROCESSED_DIR = Path("processed")

SPLIT_CSVS = {
    "train": PROCESSED_DIR / "therapistqa_train.csv",
    "val":   PROCESSED_DIR / "therapistqa_val.csv",
    "test":  PROCESSED_DIR / "therapistqa_test.csv",
}

MODEL_NAME = "roberta-base"
MAX_LENGTH = 512

OUT_LABEL_MAPS = PROCESSED_DIR / "therapistqa_bio_label_maps.json"

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
# Load the splits
splits = {name: pd.read_csv(path) for name, path in SPLIT_CSVS.items()}

for name, frame in splits.items():
    print(f"{name}: {len(frame)} records | "
          f"distorted={frame['is_distorted'].sum()} | "
          f"non-distorted={(~frame['is_distorted']).sum()}")

train: 1768 records | distorted=1116 | non-distorted=652
val: 379 records | distorted=239 | non-distorted=140
test: 379 records | distorted=239 | non-distorted=140


### Parse the all_spans column into Python lists
The column is stored as a JSON-encoded string ("[[547, 694]]" or "[[673, 763], [1457, 1706]]"); for non-distorted records it's NaN. Normalise it to a list-of-tuples and validates each span against the patient question.

In [4]:
def parse_spans(value, text):
    """
    Return a sorted list of (start, end) tuples.
    - NaN / non-distorted: returns [] (no spans).
    - Validates that 0 <= start < end <= len(text).
    - Sorts by start; the dataset has no overlapping spans (verified upstream)
    """
    if pd.isna(value):
        return []
    spans = json.loads(value)
    spans = [(int(s), int(e)) for s, e in spans]
    spans.sort(key=lambda x: x[0])
    n = len(text)
    for s, e in spans:
        assert 0 <= s < e <= n, f"Bad span [{s}, {e}] for text length {n}"
    for i in range(len(spans) - 1):
        assert spans[i][1] <= spans[i + 1][0], (
            f"Overlapping spans found: {spans[i]} and {spans[i + 1]}"
        )
    return spans

for name, frame in splits.items():
    frame["spans"]   = frame.apply(lambda r: parse_spans(r["all_spans"], r["patient_question"]), axis=1)
    frame["n_spans"] = frame["spans"].apply(len)
    print(f"{name} — spans/record:", frame["n_spans"].value_counts().sort_index().to_dict())


train — spans/record: {0: 652, 1: 977, 2: 120, 3: 17, 4: 1, 5: 1}
val — spans/record: {0: 140, 1: 207, 2: 29, 3: 3}
test — spans/record: {0: 140, 1: 207, 2: 28, 3: 4}


### Build the label vocabularies
Two label spaces, written out as JSON for downstream training scripts to load.

In [5]:
all_cats = pd.concat([f["dominant_distortion"] for f in splits.values()]).unique()
DISTORTION_CATS = sorted(c for c in all_cats if c != "No Distortion")
print("Categories used for B-/I- tags:", DISTORTION_CATS)

# 3B — 21 labels: O + B-CAT + I-CAT for each of 10 categories.
# Order: O first, then alphabetical (B,I) per category. Stable across runs.
labels_3b = ["O"]
for cat in DISTORTION_CATS:
    labels_3b.append(f"B-{cat}")
    labels_3b.append(f"I-{cat}")
label2id_3b = {lbl: i for i, lbl in enumerate(labels_3b)}
id2label_3b = {i: lbl for lbl, i in label2id_3b.items()}

# 3C — 3 labels: O, B-DIST, I-DIST
labels_3c = ["O", "B-DIST", "I-DIST"]
label2id_3c = {lbl: i for i, lbl in enumerate(labels_3c)}
id2label_3c = {i: lbl for lbl, i in label2id_3c.items()}

with open(OUT_LABEL_MAPS, "w") as f:
    json.dump(
        {
            "labels_3b": labels_3b,
            "label2id_3b": label2id_3b,
            "id2label_3b": {str(k): v for k, v in id2label_3b.items()},
            "labels_3c": labels_3c,
            "label2id_3c": label2id_3c,
            "id2label_3c": {str(k): v for k, v in id2label_3c.items()},
            "categories": DISTORTION_CATS,
        },
        f,
        indent=2,
    )

print(f"\n3B labels ({len(labels_3b)}):", labels_3b)
print(f"3C labels ({len(labels_3c)}):", labels_3c)

Categories used for B-/I- tags: ['All-or-nothing thinking', 'Emotional Reasoning', 'Fortune-telling', 'Labeling', 'Magnification', 'Mental filter', 'Mind Reading', 'Overgeneralization', 'Personalization', 'Should statements']

3B labels (21): ['O', 'B-All-or-nothing thinking', 'I-All-or-nothing thinking', 'B-Emotional Reasoning', 'I-Emotional Reasoning', 'B-Fortune-telling', 'I-Fortune-telling', 'B-Labeling', 'I-Labeling', 'B-Magnification', 'I-Magnification', 'B-Mental filter', 'I-Mental filter', 'B-Mind Reading', 'I-Mind Reading', 'B-Overgeneralization', 'I-Overgeneralization', 'B-Personalization', 'I-Personalization', 'B-Should statements', 'I-Should statements']
3C labels (3): ['O', 'B-DIST', 'I-DIST']


### Tokenise + assign BIO tags
Each record is tokenised once with `return_offsets_mapping=True`, then the (start, end) offsets returned by the tokenizer are matched against the span list to derive both the 3B and 3C tag sequences in a single pass.
The matching rule used is: a token is inside a span if its character interval is non-empty and overlaps the span; `tok_start < span_end and tok_end > span_start`.

The first token inside a given span gets B-; every subsequent token inside the same span gets I-. Special tokens (those with offset (0, 0) and is_special_tokens_mask == 1) get O and are also recorded in a separate mask so the training loop can ignore them via -100.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def assign_bio_tags(text, spans, category):
    """
    Tokenise `text` and return parallel lists:
      tokens: list[str]                   (subword strings, for inspection)
      offsets: list[tuple[int, int]]      (char offsets per token)
      tags_3b: list[str]                  (O / B-CAT / I-CAT)
      tags_3c: list[str]                  (O / B-DIST / I-DIST)
      special_mask: list[int]             (1 if special token, else 0)

    Multi-span handling: each span in `spans` independently produces a fresh B-
    at its first overlapping token; subsequent overlapping tokens get I-.
    Spans must be non-overlapping and sorted (enforced in parse_spans).
    """
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
    )
    offsets = enc["offset_mapping"]
    special_mask = enc.get("special_tokens_mask")
    if special_mask is None:
        # Fall back: a special token has offsets (0, 0)
        special_mask = [1 if (s == 0 and e == 0) else 0 for s, e in offsets]

    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])

    tags_3b = ["O"] * len(offsets)
    tags_3c = ["O"] * len(offsets)

    for span_start, span_end in spans:
        first_token_in_span = True
        for i, (tok_start, tok_end) in enumerate(offsets):
            if special_mask[i]:
                continue
            if tok_start == tok_end:  # zero-width token — skip
                continue
            # Overlap check
            if tok_start < span_end and tok_end > span_start:
                if first_token_in_span:
                    tags_3b[i] = f"B-{category}"
                    tags_3c[i] = "B-DIST"
                    first_token_in_span = False
                else:
                    tags_3b[i] = f"I-{category}"
                    tags_3c[i] = "I-DIST"
            elif tok_start >= span_end:
                # Past the end of this span; further tokens cannot overlap.
                break

    return {
        "tokens": tokens,
        "offsets": [list(o) for o in offsets],
        "tags_3b": tags_3b,
        "tags_3c": tags_3c,
        "special_mask": list(special_mask),
        "n_tokens": len(tokens),
    }


bio_dfs = {}                       # name -> DataFrame
for split_name, frame in splits.items():
    records = []
    truncated_spans = 0
    for row in frame.itertuples(index=False):
        category = row.dominant_distortion if row.is_distorted else None
        spans = row.spans
        bio = assign_bio_tags(row.patient_question, spans, category)

        # Sanity: detect spans dropped by 512-token truncation.
        if spans and bio["n_tokens"] >= MAX_LENGTH:
            last_off_end = max(
                (e for (s, e), m in zip(bio["offsets"], bio["special_mask"]) if not m),
                default=0,
            )
            if any(span_start >= last_off_end for span_start, _ in spans):
                truncated_spans += 1

        records.append({
            "id": row.id,
            "patient_question": row.patient_question,
            "dominant_distortion": row.dominant_distortion,
            "is_distorted": row.is_distorted,
            "n_spans": len(row.spans),
            "spans": row.spans,
            **bio,
        })
    bio_dfs[split_name] = pd.DataFrame(records)
    print(f"{split_name}: {len(bio_dfs[split_name])} rows | "
          f"truncation-lost spans: {truncated_spans}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

train: 1768 rows | truncation-lost spans: 18
val: 379 rows | truncation-lost spans: 2
test: 379 rows | truncation-lost spans: 2


### Verification: reconstruct spans from tags and compare
Before saving, verify the tagging round-trip. For every record, decode contiguous B-/I- runs back into character spans and check they recover the original spans (modulo truncation).

A small number of *off-by-1* mismatches is expected when a span boundary lands inside a subword; the tokenizer expands the span outward to the nearest token boundary, which is the correct behaviour for BIO tagging. The `tol=2` in `spans_match` absorbs that.

In [7]:
def tags_to_char_spans(offsets, tags_3c, special_mask):
    """Decode B-DIST / I-DIST runs back into (char_start, char_end) spans."""
    spans = []
    cur_start = None
    cur_end = None
    for (s, e), tag, m in zip(offsets, tags_3c, special_mask):
        if m or s == e:
            continue
        if tag == "B-DIST":
            if cur_start is not None:
                spans.append((cur_start, cur_end))
            cur_start, cur_end = s, e
        elif tag == "I-DIST":
            if cur_start is None:
                cur_start, cur_end = s, e   # malformed I- without B-, recover gracefully
            else:
                cur_end = e
        else:  # O
            if cur_start is not None:
                spans.append((cur_start, cur_end))
                cur_start = cur_end = None
    if cur_start is not None:
        spans.append((cur_start, cur_end))
    return spans


def spans_match(orig, decoded, tol=2):
    """Approximate match: same count + each pair within `tol` chars at both ends."""
    if len(orig) != len(decoded):
        return False
    for (s1, e1), (s2, e2) in zip(sorted(orig), sorted(decoded)):
        if abs(s1 - s2) > tol or abs(e1 - e2) > tol:
            return False
    return True


mismatches_by_split = {}

for split_name, bio_df in bio_dfs.items():
    mismatches = []
    for r in bio_df.itertuples(index=False):
        if r.n_spans == 0:
            continue
        decoded = tags_to_char_spans(r.offsets, r.tags_3c, r.special_mask)
        truncated = r.n_tokens >= MAX_LENGTH
        if not truncated and not spans_match(r.spans, decoded):
            mismatches.append((r.id, r.spans, decoded))
    mismatches_by_split[split_name] = mismatches
    print(f"{split_name}: round-trip mismatches (excl. truncation) = {len(mismatches)}")
    for mid, orig, dec in mismatches[:3]:
        print(f"  id={mid}  orig={orig}  decoded={dec}")
    print()

total = sum(len(m) for m in mismatches_by_split.values())
print(f"Total mismatches across all splits: {total}")

train: round-trip mismatches (excl. truncation) = 0

val: round-trip mismatches (excl. truncation) = 1
  id=102  orig=[(279, 310)]  decoded=[(279, 317)]

test: round-trip mismatches (excl. truncation) = 0

Total mismatches across all splits: 1


### Per-record summary statistics

In [8]:
for split_name, bio_df in bio_dfs.items():
    print(f"=== SPLIT:  {split_name} ===")
    print("=== BIO tag distribution (3C) ===")
    all_3c = [t for tags in bio_df["tags_3c"] for t in tags]
    print(pd.Series(all_3c).value_counts())

    print("\n=== BIO tag distribution (3B) — distorted-token tags only ===")
    all_3b = [t for tags in bio_df["tags_3b"] for t in tags if t != "O"]
    print(pd.Series(all_3b).value_counts())

    print("\n=== Tokens per record ===")
    print(bio_df["n_tokens"].describe())

    print("\n=== Records hitting MAX_LENGTH (512) ===")
    print((bio_df["n_tokens"] >= MAX_LENGTH).sum())

=== SPLIT:  train ===
=== BIO tag distribution (3C) ===
O         307419
I-DIST     48362
B-DIST      1258
Name: count, dtype: int64

=== BIO tag distribution (3B) — distorted-token tags only ===
I-Magnification              6740
I-Mind Reading               6374
I-Overgeneralization         5901
I-Labeling                   5547
I-Personalization            4908
I-Emotional Reasoning        4187
I-Fortune-telling            4000
I-All-or-nothing thinking    3839
I-Mental filter              3786
I-Should statements          3080
B-Overgeneralization          188
B-Mind Reading                175
B-Magnification               152
B-Labeling                    144
B-Personalization             128
B-Fortune-telling             113
B-Emotional Reasoning         103
B-Mental filter                93
B-All-or-nothing thinking      83
B-Should statements            79
Name: count, dtype: int64

=== Tokens per record ===
count    1768.000000
mean      201.945136
std       129.548870
min     

### Save
Two parquet files (one per label scheme) plus a JSONL companion for human inspection. Parquet handles list-of-strings columns natively and can be loaded via `datasets.Dataset.from_parquet(...)`.

In [9]:
common_cols = ["id", "patient_question", "dominant_distortion", "is_distorted",
               "n_spans", "spans", "tokens", "offsets", "special_mask", "n_tokens"]

def out_path(scheme: str, split: str) -> Path:
    # scheme in {"3b", "3c"};  split in {"train", "val", "test"}
    return PROCESSED_DIR / f"therapistqa_bio_{scheme}_{split}.parquet"

for split_name, bio_df in bio_dfs.items():
    bio_3b = bio_df[common_cols + ["tags_3b"]].rename(columns={"tags_3b": "tags"})
    bio_3c = bio_df[common_cols + ["tags_3c"]].rename(columns={"tags_3c": "tags"})
    bio_3b["label_ids"] = bio_3b["tags"].apply(lambda ts: [label2id_3b[t] for t in ts])
    bio_3c["label_ids"] = bio_3c["tags"].apply(lambda ts: [label2id_3c[t] for t in ts])

    bio_3b.to_parquet(out_path("3b", split_name), index=False)
    bio_3c.to_parquet(out_path("3c", split_name), index=False)
    print(f"Wrote {out_path('3b', split_name).name}  ({len(bio_3b)} rows)")
    print(f"Wrote {out_path('3c', split_name).name}  ({len(bio_3c)} rows)")

Wrote therapistqa_bio_3b_train.parquet  (1768 rows)
Wrote therapistqa_bio_3c_train.parquet  (1768 rows)
Wrote therapistqa_bio_3b_val.parquet  (379 rows)
Wrote therapistqa_bio_3c_val.parquet  (379 rows)
Wrote therapistqa_bio_3b_test.parquet  (379 rows)
Wrote therapistqa_bio_3c_test.parquet  (379 rows)


### Spot-check one multi-span record end-to-end
Quick visual check that the alignment looks right on one of the trickier records.

In [10]:
for split_name, bio_df in bio_dfs.items():
    multi = bio_df[bio_df["n_spans"] >= 2]
    if len(multi) == 0:
        print(f"=== {split_name}: no multi-span records ===\n")
        continue

    sample = multi.iloc[0]
    print(f"=== {split_name} ===")
    print(f"id={sample.id}  category={sample.dominant_distortion}")
    print(f"original spans (char): {sample.spans}")
    print(f"tokens: {sample.n_tokens}\n")

    print("token                | offset       | tag_3b                         | tag_3c")
    print("-" * 90)
    for tok, off, t3b, t3c, m in zip(
        sample.tokens, sample.offsets, sample.tags_3b, sample.tags_3c, sample.special_mask
    ):
        if m or t3c == "O":
            continue   # only show in-span rows for compactness
        print(f"{tok:20s} | {tuple(off)} | {t3b:30s} | {t3c}")
    print()

=== train ===
id=830  category=Magnification
original spans (char): [(118, 367), (692, 856)]
tokens: 279

token                | offset       | tag_3b                         | tag_3c
------------------------------------------------------------------------------------------
ĠWithin              | (118, 124) | B-Magnification                | B-DIST
Ġthe                 | (125, 128) | I-Magnification                | I-DIST
Ġpast                | (129, 133) | I-Magnification                | I-DIST
Ġfew                 | (134, 137) | I-Magnification                | I-DIST
Ġmonths              | (138, 144) | I-Magnification                | I-DIST
,                    | (144, 145) | I-Magnification                | I-DIST
ĠI                   | (146, 147) | I-Magnification                | I-DIST
âĢ                   | (147, 148) | I-Magnification                | I-DIST
Ļ                    | (147, 148) | I-Magnification                | I-DIST
ve                   | (148, 150) | I-Mag